# Quantity Field Exploration

This notebook explores the three Open Food Facts Canada quantity-related fields:

- `product_quantity_unit`
- `product_quantity`
- `quantity`

This notebook is intentionally DuckDB-first. The parquet loading, field profiling, regex pattern classification, cross-field consistency checks, and anomaly surfacing are all pushed into DuckDB SQL as much as possible.

## 1. Setup

This cell imports DuckDB and a few lightweight Python helpers. The data processing will happen in DuckDB, while Python is only used to assemble reusable SQL snippets and display query results.

In [8]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Sanjay H\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [9]:
# Import DuckDB for the actual exploration work.
import duckdb
import re
from pathlib import Path

# Import display so query results render nicely inside the notebook.
from IPython.display import display

# These are the OFF Canada fields we are exploring in this notebook.
DATA_COLUMNS = ["product_quantity_unit", "product_quantity", "quantity"]

# Surface very large numeric quantities separately because they can hide the real distribution.
LARGE_QUANTITY_THRESHOLD = 100000.0

# Try both notebook-relative locations so the notebook works from repo root or the pipeline folder.
PARQUET_CANDIDATES = [
    Path("data/raw/off-canada.parquet"),
    Path("../data/raw/off-canada.parquet"),
]

# Map raw unit spellings to a canonical base unit, quantity type, and conversion factor.
UNIT_ALIASES = [
    ("g", "g", "mass", 1.0),
    ("gram", "g", "mass", 1.0),
    ("grams", "g", "mass", 1.0),
    ("gramme", "g", "mass", 1.0),
    ("grammes", "g", "mass", 1.0),
    ("kg", "g", "mass", 1000.0),
    ("kilo", "g", "mass", 1000.0),
    ("kilos", "g", "mass", 1000.0),
    ("kilogram", "g", "mass", 1000.0),
    ("kilograms", "g", "mass", 1000.0),
    ("mg", "g", "mass", 0.001),
    ("ml", "ml", "volume", 1.0),
    ("m l", "ml", "volume", 1.0),
    ("l", "ml", "volume", 1000.0),
    ("liter", "ml", "volume", 1000.0),
    ("liters", "ml", "volume", 1000.0),
    ("litre", "ml", "volume", 1000.0),
    ("litres", "ml", "volume", 1000.0),
    ("cl", "ml", "volume", 10.0),
    ("oz", "g", "mass", 28.349523125),
    ("lb", "g", "mass", 453.59237),
    ("lbs", "g", "mass", 453.59237),
    ("fl oz", "ml", "volume", 29.5735),
    ("pc", "count", "count", 1.0),
    ("pcs", "count", "count", 1.0),
    ("piece", "count", "count", 1.0),
    ("pieces", "count", "count", 1.0),
    ("tablet", "count", "count", 1.0),
    ("tablets", "count", "count", 1.0),
    ("caps", "count", "count", 1.0),
    ("cap", "count", "count", 1.0),
    ("capsule", "count", "count", 1.0),
    ("capsules", "count", "count", 1.0),
    ("egg", "count", "count", 1.0),
    ("eggs", "count", "count", 1.0),
    ("bag", "count", "count", 1.0),
    ("bags", "count", "count", 1.0),
    ("bar", "count", "count", 1.0),
    ("bars", "count", "count", 1.0),
    ("barre", "count", "count", 1.0),
    ("barres", "count", "count", 1.0),
    ("portion", "count", "count", 1.0),
    ("portions", "count", "count", 1.0),
    ("can", "count", "count", 1.0),
    ("cans", "count", "count", 1.0),
    ("mint", "count", "count", 1.0),
    ("mints", "count", "count", 1.0),
    ("sachet", "count", "count", 1.0),
    ("sachets", "count", "count", 1.0),
    ("bottle", "count", "count", 1.0),
    ("bottles", "count", "count", 1.0),
    ("bun", "count", "count", 1.0),
    ("buns", "count", "count", 1.0),
    ("yaourt", "count", "count", 1.0),
    ("yaourts", "count", "count", 1.0),
    ("tranche", "count", "count", 1.0),
    ("tranches", "count", "count", 1.0),
    ("serving", "count", "count", 1.0),
    ("servings", "count", "count", 1.0),
]

# These raw tokens usually indicate the free-text quantity is not directly useful yet.
PLACEHOLDER_REGEX = r"(unknown|bonne|good|bn batouta)"

# These count-like words help us separate measure rows from count-descriptor rows.
COUNT_DESCRIPTOR_REGEX = r"\b(count|pc|pcs|piece|pieces|tablet|tablets|caps|cap|capsule|capsules|egg|eggs|bag|bags|bar|bars|barre|barres|portion|portions|can|cans|mint|mints|sachet|sachets|serving|servings)\b"

# Ignore these leftover tokens when surfacing unresolved words.
UNRESOLVED_TOKEN_STOPWORDS = [
    "and", "de", "des", "du", "en", "et", "in", "net", "wt",
    "quot", "the", "of", "pour", "double"
]


# Resolve the parquet path once so the notebook can run from multiple working directories.
def resolve_parquet_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find OFF parquet. Tried: {candidates}")


# Escape a Python string so it is safe to embed inside SQL string literals.
def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


# Build one regex pattern covering all currently recognized unit tokens.
def build_unit_pattern(unit_aliases):
    tokens = [token for token, _, _, _ in unit_aliases]
    return "(?:" + "|".join(re.escape(token).replace(r"\ ", r"\s*") for token in sorted(tokens, key=len, reverse=True)) + ")"


# Build a reusable SQL CASE expression that maps a raw token to its canonical unit metadata.
def build_unit_case(expr, target):
    target_index = {"base_unit": 1, "quantity_type": 2, "factor": 3}[target]
    lines = ["CASE"]
    for token, base_unit, quantity_type, factor in UNIT_ALIASES:
        mapped_value = [token, base_unit, quantity_type, factor][target_index]
        sql_value = str(mapped_value) if target == "factor" else sql_string(mapped_value)
        lines.append(f"    WHEN {expr} = {sql_string(token)} THEN {sql_value}")
    lines.append("    ELSE NULL")
    lines.append("END")
    return "\n".join(lines)


# Small helper so every query cell can render a titled result table consistently.
def show_query(title, query):
    print(title)
    display(con.sql(query).df())


UNIT_PATTERN = build_unit_pattern(UNIT_ALIASES)
SIMPLE_MEASURE_REGEX = rf"^([0-9]+(?:[.,][0-9]+)?)\s*({UNIT_PATTERN})$"
MULTIPACK_REGEX = rf"\b(\d+)\s*[*x]\s*([0-9]+(?:[.,][0-9]+)?)\s*({UNIT_PATTERN})\b"
GENERAL_UNIT_REGEX = rf"({UNIT_PATTERN})"

RECOGNIZED_UNIT_TOKENS_SQL = ", ".join(sql_string(token) for token, _, _, _ in UNIT_ALIASES)
UNRESOLVED_STOPWORDS_SQL = ", ".join(sql_string(token) for token in UNRESOLVED_TOKEN_STOPWORDS)

# Create the in-memory DuckDB connection once for the whole notebook.
con = duckdb.connect()


## 2. Load The Quantity Fields Into DuckDB

This cell reads the parquet through DuckDB and creates two views:

- `quantity_raw`: the three original OFF quantity fields only
- `quantity_features`: a DuckDB-derived profiling view with normalized text, parsed values, canonical units, and pattern labels

The notebook will query these views directly instead of pushing the analysis into pandas.

In [10]:
# Resolve the parquet path and expose only the three relevant fields as a DuckDB view.
PARQUET_PATH = resolve_parquet_path(PARQUET_CANDIDATES)
PARQUET_SQL_PATH = PARQUET_PATH.as_posix().replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW quantity_raw AS
    SELECT {', '.join(DATA_COLUMNS)}
    FROM read_parquet('{PARQUET_SQL_PATH}')
    """
)

# Build reusable SQL CASE expressions for unit normalization.
PRODUCT_UNIT_BASE_CASE = build_unit_case("product_quantity_unit_normalized", "base_unit")
PRODUCT_UNIT_TYPE_CASE = build_unit_case("product_quantity_unit_normalized", "quantity_type")
SIMPLE_UNIT_BASE_CASE = build_unit_case("simple_unit_token", "base_unit")
SIMPLE_UNIT_TYPE_CASE = build_unit_case("simple_unit_token", "quantity_type")
SIMPLE_UNIT_FACTOR_CASE = build_unit_case("simple_unit_token", "factor")
PACK_UNIT_BASE_CASE = build_unit_case("pack_unit_token", "base_unit")
PACK_UNIT_TYPE_CASE = build_unit_case("pack_unit_token", "quantity_type")
PACK_UNIT_FACTOR_CASE = build_unit_case("pack_unit_token", "factor")
FIRST_UNIT_BASE_CASE = build_unit_case("first_unit_token", "base_unit")
FIRST_UNIT_TYPE_CASE = build_unit_case("first_unit_token", "quantity_type")

# Create one derived DuckDB view that holds the parsed and normalized exploration features.
quantity_features_sql = f"""
CREATE OR REPLACE TEMP VIEW quantity_features AS
WITH base AS (
    SELECT
        product_quantity_unit,
        lower(trim(product_quantity_unit)) AS product_quantity_unit_normalized,
        product_quantity,
        TRY_CAST(product_quantity AS DOUBLE) AS product_quantity_numeric,
        quantity,
        lower(trim(quantity)) AS quantity_normalized
    FROM quantity_raw
),
parsed AS (
    SELECT
        *,
        NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 1), '') AS simple_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 2), '') AS simple_unit_token,
        TRY_CAST(replace(NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 1), ''), ',', '.') AS DOUBLE) AS simple_value_raw,
        TRY_CAST(NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_REGEX}', 1), '') AS DOUBLE) AS pack_count,
        TRY_CAST(replace(NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_REGEX}', 2), ''), ',', '.') AS DOUBLE) AS pack_value_raw,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_REGEX}', 3), '') AS pack_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{GENERAL_UNIT_REGEX}', 1), '') AS first_unit_token
    FROM base
),
normalized AS (
    SELECT
        *,
        {PRODUCT_UNIT_BASE_CASE} AS product_quantity_unit_base,
        {PRODUCT_UNIT_TYPE_CASE} AS product_quantity_unit_type,
        {SIMPLE_UNIT_BASE_CASE} AS simple_base_unit,
        {SIMPLE_UNIT_TYPE_CASE} AS simple_quantity_type,
        {SIMPLE_UNIT_FACTOR_CASE} AS simple_factor,
        CASE
            WHEN simple_value_raw IS NOT NULL AND {SIMPLE_UNIT_FACTOR_CASE} IS NOT NULL
                THEN simple_value_raw * {SIMPLE_UNIT_FACTOR_CASE}
            ELSE NULL
        END AS simple_normalized_value,
        {PACK_UNIT_BASE_CASE} AS pack_base_unit,
        {PACK_UNIT_TYPE_CASE} AS pack_quantity_type,
        {PACK_UNIT_FACTOR_CASE} AS pack_factor,
        CASE
            WHEN pack_count IS NOT NULL AND pack_value_raw IS NOT NULL AND {PACK_UNIT_FACTOR_CASE} IS NOT NULL
                THEN pack_count * pack_value_raw * {PACK_UNIT_FACTOR_CASE}
            ELSE NULL
        END AS pack_normalized_total_value,
        {FIRST_UNIT_BASE_CASE} AS first_unit_base,
        {FIRST_UNIT_TYPE_CASE} AS first_quantity_type
    FROM parsed
)
SELECT
    *,
    CASE
        WHEN quantity IS NULL THEN 'null'
        WHEN quantity_normalized = '' THEN 'blank'
        WHEN regexp_matches(quantity_normalized, '{PLACEHOLDER_REGEX}') THEN 'placeholder_or_unknown'
        WHEN simple_value_raw IS NOT NULL AND simple_factor IS NOT NULL THEN 'simple_measure'
        WHEN pack_count IS NOT NULL AND pack_value_raw IS NOT NULL AND pack_factor IS NOT NULL THEN 'multipack_measure'
        WHEN regexp_matches(quantity_normalized, '^[0-9]+(?:[.,][0-9]+)?$') THEN 'number_only'
        WHEN regexp_matches(quantity_normalized, '\\b\\d+\\s*[*x]\\s*\\d+') THEN 'multipack_unparsed'
        WHEN regexp_matches(quantity_normalized, '[()/|]') AND first_unit_token IS NOT NULL THEN 'mixed_measure_expression'
        WHEN first_unit_token IS NOT NULL AND regexp_matches(quantity_normalized, '\\d') THEN 'text_with_measure'
        WHEN regexp_matches(quantity_normalized, '{COUNT_DESCRIPTOR_REGEX}') THEN 'count_descriptor'
        ELSE 'unparsed_text'
    END AS quantity_pattern
FROM normalized
"""

con.execute(quantity_features_sql)

# Show the parquet source, row count, schema, and a few sample rows from the raw view.
print(f"Source parquet: {PARQUET_PATH}")
show_query("Raw schema", "DESCRIBE quantity_raw")
show_query("Row count", "SELECT COUNT(*) AS row_count FROM quantity_raw")
show_query("Sample rows", "SELECT * FROM quantity_raw LIMIT 10")


Source parquet: ..\data\raw\off-canada.parquet
Raw schema


,column_name,column_type,null,key,default,extra
0,product_quantity_unit,VARCHAR,YES,None,None,None
1,product_quantity,VARCHAR,YES,None,None,None
2,quantity,VARCHAR,YES,None,None,None


Row count


,row_count
0,114453


Sample rows


,product_quantity_unit,product_quantity,quantity
0,ml,946,946 ml
1,None,118,118 ml
2,None,None,
3,None,235,235 g
4,None,0,50
5,g,50,50g
6,g,192,192 g
7,None,None,None
8,None,None,None
9,ml,480,480 mL


## 3. Field Coverage Profile

This cell gives the completeness picture for all three fields: null counts, blank-string counts, non-blank counts, and distinct-value counts. Because the work is in DuckDB, these numbers come directly from SQL aggregations over the parquet-backed view.

In [11]:
# Compare the three quantity fields side by side using one SQL query.
coverage_query = """
SELECT 'product_quantity_unit' AS field,
       COUNT(*) AS row_count,
       COUNT(*) FILTER (WHERE product_quantity_unit IS NULL) AS null_count,
       COUNT(*) FILTER (WHERE product_quantity_unit IS NOT NULL) AS non_null_count,
       COUNT(*) FILTER (WHERE product_quantity_unit IS NOT NULL AND trim(product_quantity_unit) = '') AS blank_string_count,
       COUNT(*) FILTER (WHERE product_quantity_unit IS NOT NULL AND trim(product_quantity_unit) <> '') AS non_blank_count,
       COUNT(DISTINCT product_quantity_unit) FILTER (WHERE product_quantity_unit IS NOT NULL AND trim(product_quantity_unit) <> '') AS distinct_non_blank_count
FROM quantity_raw
UNION ALL
SELECT 'product_quantity' AS field,
       COUNT(*) AS row_count,
       COUNT(*) FILTER (WHERE product_quantity IS NULL) AS null_count,
       COUNT(*) FILTER (WHERE product_quantity IS NOT NULL) AS non_null_count,
       COUNT(*) FILTER (WHERE product_quantity IS NOT NULL AND trim(product_quantity) = '') AS blank_string_count,
       COUNT(*) FILTER (WHERE product_quantity IS NOT NULL AND trim(product_quantity) <> '') AS non_blank_count,
       COUNT(DISTINCT product_quantity) FILTER (WHERE product_quantity IS NOT NULL AND trim(product_quantity) <> '') AS distinct_non_blank_count
FROM quantity_raw
UNION ALL
SELECT 'quantity' AS field,
       COUNT(*) AS row_count,
       COUNT(*) FILTER (WHERE quantity IS NULL) AS null_count,
       COUNT(*) FILTER (WHERE quantity IS NOT NULL) AS non_null_count,
       COUNT(*) FILTER (WHERE quantity IS NOT NULL AND trim(quantity) = '') AS blank_string_count,
       COUNT(*) FILTER (WHERE quantity IS NOT NULL AND trim(quantity) <> '') AS non_blank_count,
       COUNT(DISTINCT quantity) FILTER (WHERE quantity IS NOT NULL AND trim(quantity) <> '') AS distinct_non_blank_count
FROM quantity_raw
"""

show_query("Coverage summary", coverage_query)


Coverage summary


,field,row_count,null_count,non_null_count,blank_string_count,non_blank_count,distinct_non_blank_count
0,product_quantity_unit,114453,98638,15815,0,15815,3
1,product_quantity,114453,93022,21431,0,21431,1652
2,quantity,114453,90285,24168,2416,21752,4362


## 4. Explore `product_quantity_unit`

This cell focuses only on the normalized unit field. We want to know how sparse it is, which values dominate it, and whether any unexpected unit tokens are present.

In [12]:
# Show the most common raw values in product_quantity_unit.
show_query(
    "Top raw product_quantity_unit values",
    """
    SELECT COALESCE(product_quantity_unit, '<NA>') AS value, COUNT(*) AS count
    FROM quantity_raw
    GROUP BY 1
    ORDER BY count DESC, value
    LIMIT 20
    """
)

# Map the raw unit tokens to canonical base units so we can see the real unit distribution.
show_query(
    "Canonical base unit counts from product_quantity_unit",
    f"""
    SELECT COALESCE(product_quantity_unit_base, '__unrecognized__') AS canonical_base_unit,
           COUNT(*) AS count
    FROM quantity_features
    WHERE product_quantity_unit IS NOT NULL AND trim(product_quantity_unit) <> ''
    GROUP BY 1
    ORDER BY count DESC, canonical_base_unit
    """
)

# Surface any raw unit tokens that we are not recognizing yet.
show_query(
    "Unrecognized product_quantity_unit values",
    """
    SELECT DISTINCT product_quantity_unit AS unrecognized_unit
    FROM quantity_features
    WHERE product_quantity_unit IS NOT NULL
      AND trim(product_quantity_unit) <> ''
      AND product_quantity_unit_base IS NULL
    ORDER BY unrecognized_unit
    """
)


Top raw product_quantity_unit values


,value,count
0,<NA>,98638
1,g,11813
2,ml,4001
3,kj,1


Canonical base unit counts from product_quantity_unit


,canonical_base_unit,count
0,g,11813
1,ml,4001
2,__unrecognized__,1


Unrecognized product_quantity_unit values


,unrecognized_unit
0,kj


## 5. Explore `product_quantity`

This cell profiles the numeric quantity field. The query checks how often it is populated, whether it is numeric when present, how often zero appears, what the typical range looks like, and whether a few extreme rows distort the mean.

In [13]:
# Build a robust numeric profile directly in SQL.
product_quantity_profile_query = f"""
    WITH numeric_base AS (
        SELECT product_quantity,
               product_quantity_numeric
        FROM quantity_features
    ),
    thresholds AS (
        SELECT quantile_cont(product_quantity_numeric, 0.99) AS p99
        FROM numeric_base
        WHERE product_quantity_numeric IS NOT NULL
    )
    SELECT
        COUNT(*) FILTER (WHERE product_quantity IS NOT NULL) AS non_null_count,
        COUNT(*) FILTER (WHERE product_quantity_numeric IS NOT NULL) AS numeric_cast_count,
        COUNT(*) FILTER (WHERE product_quantity IS NOT NULL AND product_quantity_numeric IS NULL) AS numeric_cast_failure_count,
        COUNT(*) FILTER (WHERE product_quantity_numeric = 0) AS zero_count,
        COUNT(*) FILTER (WHERE product_quantity_numeric < 0) AS negative_count,
        COUNT(*) FILTER (WHERE product_quantity_numeric > 0) AS positive_count,
        COUNT(*) FILTER (WHERE product_quantity_numeric IS NOT NULL AND product_quantity_numeric = floor(product_quantity_numeric)) AS integer_like_count,
        COUNT(*) FILTER (WHERE product_quantity_numeric IS NOT NULL AND product_quantity_numeric <> floor(product_quantity_numeric)) AS fractional_count,
        MIN(product_quantity_numeric) AS min_value,
        MIN(product_quantity_numeric) FILTER (WHERE product_quantity_numeric > 0) AS positive_min_value,
        quantile_cont(product_quantity_numeric, 0.05) AS p5,
        median(product_quantity_numeric) AS median,
        AVG(product_quantity_numeric) AS mean,
        AVG(product_quantity_numeric) FILTER (WHERE product_quantity_numeric <= p99) AS trimmed_mean_excluding_top_1_percent,
        quantile_cont(product_quantity_numeric, 0.95) AS p95,
        MAX(p99) AS p99,
        MAX(product_quantity_numeric) AS max_value,
        COUNT(*) FILTER (WHERE product_quantity_numeric > {LARGE_QUANTITY_THRESHOLD}) AS rows_above_100000,
        COUNT(*) FILTER (WHERE product_quantity_numeric > p99) AS rows_above_p99
    FROM numeric_base
    CROSS JOIN thresholds
    WHERE product_quantity_numeric IS NOT NULL
"""

show_query("Numeric profile for product_quantity", product_quantity_profile_query)

# Show the most common raw product_quantity values because repeated sizes and repeated zeros matter later.
show_query(
    "Top raw product_quantity values",
    """
    SELECT COALESCE(product_quantity, '<NA>') AS value, COUNT(*) AS count
    FROM quantity_raw
    GROUP BY 1
    ORDER BY count DESC, value
    LIMIT 20
    """
)

# Surface the largest rows explicitly so extreme outliers do not stay hidden in the summary stats.
show_query(
    "Large product_quantity examples",
    f"""
    SELECT product_quantity, quantity
    FROM quantity_features
    WHERE product_quantity_numeric > {LARGE_QUANTITY_THRESHOLD}
    ORDER BY product_quantity_numeric DESC
    """
)


Numeric profile for product_quantity


,non_null_count,numeric_cast_count,numeric_cast_failure_count,zero_count,negative_count,positive_count,integer_like_count,fractional_count,min_value,positive_min_value,p5,median,mean,trimmed_mean_excluding_top_1_percent,p95,p99,max_value,rows_above_100000,rows_above_p99
0,21431,21431,0,738,0,20693,20609,822,0.0,0.000062,24.0,354.0,4.666138e+13,484.149003,1750.0,4056.0,1.000000e+18,3,215


Top raw product_quantity values


,value,count
0,<NA>,93022
1,500,725
2,1000,572
3,0,518
4,400,452
5,200,417
6,100,409
7,300,390
8,250,375
9,454,278


Large product_quantity examples


,product_quantity,quantity
0,999999999999999999,999999999999999999 g
1,1250000,1250 kg
2,400000,1 400 kg


## 6. Explore `quantity`

This cell profiles the free-text quantity field. The derived view has already labeled each row into one pattern bucket, so here we can inspect the variation directly from SQL: top raw values, pattern counts, implied base units, and example rows by pattern.

In [14]:
# Show the most common raw values in the free-text quantity field.
show_query(
    "Top raw quantity values",
    """
    SELECT COALESCE(quantity, '<NA>') AS value, COUNT(*) AS count
    FROM quantity_raw
    GROUP BY 1
    ORDER BY count DESC, value
    LIMIT 20
    """
)

# Count how many rows fall into each derived pattern bucket.
show_query(
    "Quantity pattern counts",
    """
    SELECT quantity_pattern, COUNT(*) AS count
    FROM quantity_features
    GROUP BY 1
    ORDER BY count DESC, quantity_pattern
    """
)

# Count the implied canonical base units coming from the free-text quantity field.
show_query(
    "Recognized base units from quantity",
    """
    SELECT COALESCE(
               CASE
                   WHEN simple_base_unit IS NOT NULL THEN simple_base_unit
                   WHEN pack_base_unit IS NOT NULL THEN pack_base_unit
                   ELSE first_unit_base
               END,
               '<NA>'
           ) AS recognized_base_unit,
           COUNT(*) AS count
    FROM quantity_features
    GROUP BY 1
    ORDER BY count DESC, recognized_base_unit
    """
)

# Keep a few example rows per pattern so the variation is visible without scanning the whole dataset.
show_query(
    "Sample quantity values by pattern",
    """
    WITH ranked AS (
        SELECT quantity_pattern,
               quantity,
               ROW_NUMBER() OVER (PARTITION BY quantity_pattern ORDER BY quantity) AS rn
        FROM quantity_features
        WHERE quantity IS NOT NULL
    )
    SELECT quantity_pattern, quantity
    FROM ranked
    WHERE rn <= 10
    ORDER BY quantity_pattern, rn
    """
)


Top raw quantity values


,value,count
0,<NA>,90285
1,,2416
2,500 g,421
3,100 g,419
4,200 g,367
5,400 g,337
6,300 g,316
7,454 g,306
8,1 kg,280
9,250 g,234


Quantity pattern counts


,quantity_pattern,count
0,null,90285
1,simple_measure,17373
2,multipack_measure,2734
3,blank,2416
4,number_only,577
5,text_with_measure,569
6,mixed_measure_expression,284
7,unparsed_text,191
8,multipack_unparsed,12
9,placeholder_or_unknown,9


Recognized base units from quantity


,recognized_base_unit,count
0,<NA>,93438
1,g,15785
2,ml,4932
3,count,298


Sample quantity values by pattern


,quantity_pattern,quantity
0,blank,
1,blank,
2,blank,
3,blank,
4,blank,
...,...,...
87,unparsed_text,068488 075279
88,unparsed_text,1 23 i puł
89,unparsed_text,1 Dry Pint
90,unparsed_text,1 EA


## 7. Cross-Field Relationships

This cell compares the three fields together. It shows which fields co-occur, how many missing-unit rows look recoverable from the free-text quantity, and how often the numeric quantity agrees with the parseable raw quantity strings.

In [16]:
# Show the overlap pattern across the three quantity fields.
show_query(
    "Presence combinations across the three fields",
    """
    SELECT
        product_quantity_unit IS NOT NULL AS product_quantity_unit_present,
        product_quantity IS NOT NULL AS product_quantity_present,
        quantity IS NOT NULL AS quantity_present,
        COUNT(*) AS rows
    FROM quantity_raw
    GROUP BY 1, 2, 3
    ORDER BY rows DESC
    """
)

# Summarize the key relationship counts we need before defining cleaning rules.
cross_field_summary_query = """
    SELECT 'rows_with_blank_quantity_string' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE quantity IS NOT NULL AND quantity_normalized = ''
    UNION ALL
    SELECT 'rows_with_product_quantity_zero' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE product_quantity_numeric = 0
    UNION ALL
    SELECT 'rows_with_numeric_and_missing_unit' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE product_quantity_numeric IS NOT NULL
      AND (product_quantity_unit IS NULL OR trim(product_quantity_unit) = '')
    UNION ALL
    SELECT 'rows_with_numeric_and_missing_unit_recoverable_from_quantity' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE product_quantity_numeric IS NOT NULL
      AND (product_quantity_unit IS NULL OR trim(product_quantity_unit) = '')
      AND (simple_base_unit IS NOT NULL OR pack_base_unit IS NOT NULL OR first_unit_base IS NOT NULL)
    UNION ALL
    SELECT 'rows_with_only_quantity' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE product_quantity_numeric IS NULL
      AND quantity IS NOT NULL
      AND quantity_normalized <> ''
    UNION ALL
    SELECT 'rows_with_only_quantity_simple_or_multipack_parse' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE product_quantity_numeric IS NULL
      AND quantity_pattern IN ('simple_measure', 'multipack_measure')
    UNION ALL
    SELECT 'simple_or_multipack_rows' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE quantity_pattern IN ('simple_measure', 'multipack_measure')
    UNION ALL
    SELECT 'simple_or_multipack_value_match_count' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE quantity_pattern IN ('simple_measure', 'multipack_measure')
      AND product_quantity_numeric IS NOT NULL
      AND product_quantity_numeric = COALESCE(simple_normalized_value, pack_normalized_total_value)
    UNION ALL
    SELECT 'simple_or_multipack_unit_match_count' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE quantity_pattern IN ('simple_measure', 'multipack_measure')
      AND product_quantity_unit_base IS NOT NULL
      AND product_quantity_unit_base = COALESCE(simple_base_unit, pack_base_unit)
    UNION ALL
    SELECT 'simple_or_multipack_value_mismatch_count' AS metric,
           COUNT(*) AS value
    FROM quantity_features
    WHERE quantity_pattern IN ('simple_measure', 'multipack_measure')
      AND product_quantity_numeric IS NOT NULL
      AND product_quantity_numeric <> COALESCE(simple_normalized_value, pack_normalized_total_value)
"""

show_query("Cross-field summary metrics", cross_field_summary_query)

# Show recoverable missing-unit rows so we can verify what a future backfill would look like.
show_query(
    "Recoverable missing-unit examples",
    """
    SELECT product_quantity, quantity,
           COALESCE(simple_base_unit, pack_base_unit, first_unit_base) AS inferred_base_unit
    FROM quantity_features
    WHERE product_quantity_numeric IS NOT NULL
      AND (product_quantity_unit IS NULL OR trim(product_quantity_unit) = '')
      AND COALESCE(simple_base_unit, pack_base_unit, first_unit_base) IS NOT NULL
    LIMIT 100
    """
)

# Show mismatches so later cleaning logic can treat them carefully instead of assuming all rows agree.
show_query(
    "Simple or multipack value mismatch examples",
    """
    SELECT product_quantity_unit,
           product_quantity,
           quantity,
           COALESCE(simple_normalized_value, pack_normalized_total_value) AS parsed_normalized_quantity
    FROM quantity_features
    WHERE quantity_pattern IN ('simple_measure', 'multipack_measure')
      AND product_quantity_numeric IS NOT NULL
      AND product_quantity_numeric <> COALESCE(simple_normalized_value, pack_normalized_total_value)
    LIMIT 20
    """
)


Presence combinations across the three fields


,product_quantity_unit_present,product_quantity_present,quantity_present,rows
0,False,False,False,90285
1,True,True,True,15815
2,False,True,True,5616
3,False,False,True,2737


Cross-field summary metrics


,metric,value
0,rows_with_blank_quantity_string,2416
1,rows_with_product_quantity_zero,738
2,rows_with_numeric_and_missing_unit,5616
3,rows_with_numeric_and_missing_unit_recoverable_from_quantity,5159
4,rows_with_only_quantity,321
5,rows_with_only_quantity_simple_or_multipack_parse,16
6,simple_or_multipack_rows,20107
7,simple_or_multipack_value_match_count,19851
8,simple_or_multipack_unit_match_count,15117
9,simple_or_multipack_value_mismatch_count,240


Recoverable missing-unit examples


,product_quantity,quantity,inferred_base_unit
0,118,118 ml,ml
1,235,235 g,g
2,3,3 grammes,g
3,0,180 g,g
4,0.0,1pcs,count
...,...,...,...
95,175,175 grammes,g
96,1560,1.56 kg,g
97,750,750 g,g
98,480,4 x 120 g,g


Simple or multipack value mismatch examples


,product_quantity_unit,product_quantity,quantity,parsed_normalized_quantity
0,None,0,180 g,180.00000
1,None,0.0,1pcs,1.00000
2,None,0.0,32 portions,32.00000
3,None,1359.9999999999998,1.36kg,1360.00000
4,None,0.0,12 yaourts,12.00000
5,None,0.0,10 bags,10.00000
6,None,1890.0000000000002,1.89 l,1890.00000
7,None,0,100 tablets,100.00000
8,None,0.0,1 can,1.00000
9,None,0,20 sachets,20.00000


## 8. Anomalies And Unresolved Tokens

This cell surfaces the rows and leftover raw text fragments that still need explicit cleaning rules later. The goal here is not to clean them yet, but to make sure the next step starts from concrete evidence instead of guesswork.

In [17]:
# Show zero-valued product_quantity rows with their raw quantity text.
show_query(
    "Zero product_quantity examples",
    """
    SELECT product_quantity, quantity
    FROM quantity_features
    WHERE product_quantity_numeric = 0
      AND quantity IS NOT NULL
    LIMIT 20
    """
)

# Surface the most common leftover tokens from unresolved quantity strings.
show_query(
    "Top unresolved tokens from hard-to-parse quantity rows",
    f"""
    WITH tokens AS (
        SELECT token
        FROM quantity_features,
             UNNEST(regexp_extract_all(COALESCE(quantity_normalized, ''), '[a-z]+')) AS token_table(token)
        WHERE quantity_pattern IN ('unparsed_text', 'multipack_unparsed', 'placeholder_or_unknown', 'count_descriptor')
    )
    SELECT token AS value, COUNT(*) AS count
    FROM tokens
    WHERE length(token) > 1
      AND token NOT IN ({RECOGNIZED_UNIT_TOKENS_SQL})
      AND token NOT IN ({UNRESOLVED_STOPWORDS_SQL})
    GROUP BY 1
    ORDER BY count DESC, value
    LIMIT 20
    """
)

# Show unresolved quantity rows directly so later cleaning rules can be based on real examples.
show_query(
    "Sample unresolved quantity rows",
    """
    SELECT quantity_pattern, quantity
    FROM quantity_features
    WHERE quantity_pattern IN ('unparsed_text', 'multipack_unparsed', 'placeholder_or_unknown', 'count_descriptor')
    LIMIT 30
    """
)


Zero product_quantity examples


,product_quantity,quantity
0,0,50
1,0,180 g
2,0.0,1pcs
3,0.0,6 Tablets / Tablillas
4,0,1
5,0.0,1
6,0.0,32 portions
7,0,6
8,0.0,4
9,0.0,450


Top unresolved tokens from hard-to-parse quantity rows


,value,count
0,per,9
1,qty,6
2,comprim,5
3,morceaux,5
4,quantity,5
5,unknown,5
6,chips,3
7,dry,3
8,gr,3
9,pack,3


Sample unresolved quantity rows


,quantity_pattern,quantity
0,placeholder_or_unknown,Unknown Quantity
1,placeholder_or_unknown,Unknown Quantity
2,unparsed_text,12 morceaux
3,unparsed_text,1/2
4,unparsed_text,14 tasses
5,unparsed_text,1.13 k
6,unparsed_text,1 tsp
7,placeholder_or_unknown,Bonne
8,placeholder_or_unknown,bn batouta
9,placeholder_or_unknown,Good


## 9. Ready For The Cleaning Stage

After running the notebook, we should have the main information needed for the next stage:

- completeness and sparsity of each field
- actual unit vocabulary and unrecognized unit outliers
- numeric behavior and outliers in `product_quantity`
- raw text patterns and unresolved variants in `quantity`
- overlap and consistency across all three fields
- real examples that will drive the cleaning rules later

The next step can now focus on cleaning design instead of first-pass discovery.